In [ ]:

import os
from google.colab import files

os.makedirs('./data', exist_ok=True)
print('upload seismic-bumpsdata.arff:')
uploaded = files.upload()

for filename in uploaded:
    os.rename(filename, f'./data/{filename}')
    print("file is in data directory")

In [ ]:
import numpy as np
import pandas as pd
from scipy.io import arff
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
import joblib

def plot(pandaframe):
    pandaframe.hist(bins=50, figsize=(20,15))
    plt.savefig("attribute_histogram_plots")
    plt.show()

def getBursts(pandaframe):
    rockbursts = []
    count = 0

    for index, bclass in enumerate(pandaframe['class']):
        if bclass == b'1':
            rockbursts.append(index)
    for index in rockbursts:
        print(pandaframe.iloc[index])
        count = count + 1

    print(count)

def prepareData(pandaframe):
    cat_to_num(pandaframe)
    xtrain, x_test, y_train, y_test = split_train_test(pandaframe)
    return normalizedata(xtrain, x_test, y_train, y_test)

#done right before training
def normalizedata(x_train, x_test, y_train, y_test):
    x_train = x_train.drop(['nbumps6', 'nbumps7', 'nbumps89'], axis=1)
    x_test = x_test.drop(['nbumps6', 'nbumps7', 'nbumps89'], axis=1)
    skewed = ['genergy', 'gpuls', 'energy', 'maxenergy']
    # replaces 0s with an extremely small float
    epsilon = np.finfo(float).eps
    for col in skewed:
        x_train[col] = np.log(x_train[col].replace(0, epsilon))
        x_test[col] = np.log(x_test[col].replace(0, epsilon))

    numericals = ['seismic', 'seismoacoustic', 'ghazard', 'genergy', 'gpuls', 'gdenergy', 'gdpuls', 'ghazard',
                  'nbumps', 'nbumps2', 'nbumps3', 'nbumps4', 'energy',
                  'maxenergy']
    normalizer = ColumnTransformer(transformers=[
        ('num', RobustScaler(), numericals),
        ('catagoric', OneHotEncoder(sparse_output=False), ['shift'])
    ])



    x_train_norm = normalizer.fit_transform(x_train)
    joblib.dump(normalizer, 'normalizer.pk1')
    x_test_norm = normalizer.transform(x_test)

    #smote = SMOTE(sampling_strategy=0.5,random_state=42)
    #x_train_full, y_train_full = smote.fit_resample(x_train_norm, y_train)

    #return x_train_full, x_test_norm, y_train_full.to_numpy(), y_test.to_numpy()
    return x_train_norm, x_test_norm, y_train.to_numpy(), y_test.to_numpy()

def split_train_test(df):
    x = df.drop('class', axis=1)
    y = df['class']

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    return x_train, x_test, y_train, y_test

def cat_to_num(df):
    df['class'] = df['class'].map({b'1': 1, b'0': 0})

    hazard_map = {b'a': 0, b'b': 1, b'c': 2, b'd': 3}
    df['seismic'] = df['seismic'].map(hazard_map)
    df['seismoacoustic'] = df['seismoacoustic'].map(hazard_map)
    df['ghazard'] = df['ghazard'].map(hazard_map)
    df['shift'] = df['shift'].map({b'W': 0, b'N': 1})

def compute_weights(y_train):
    weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
    class_weight_dict = {0: weights[0], 1: weights[1]}
    return class_weight_dict

arff_file = arff.loadarff('./data/seismic-bumpsdata.arff')

pandaframe = pd.DataFrame(arff_file[0])

#rockframe = pandaframe[pandaframe['class'] == b'1']
#plot(rockframe)

x_train, x_test, y_train, y_test = prepareData(pandaframe)
#print(x_test)

#print(pandaframe)

#getBursts(pandaframe)
#plot(pandaframe)


In [ ]:
import tensorflow as tf
from tensorflow import keras
def main():
  class_weights_dict = compute_weights(y_train)
  print(class_weights_dict)

  model = keras.models.Sequential()

  model.add(keras.layers.Input(shape=(x_train.shape[1],)))

  model.add(keras.layers.Dense(8, activation='relu'))
  model.add(keras.layers.BatchNormalization())
  model.add(keras.layers.Dropout(0.4))

  model.add(keras.layers.Dense(4, activation='relu'))
  model.add(keras.layers.BatchNormalization())

  model.add(keras.layers.Dense(1, activation='sigmoid'))
  keras.backend.clear_session()
  np.random.seed(42)
  tf.random.set_seed(42)

  early_stop = keras.callbacks.EarlyStopping(
  monitor='val_recall',
  patience=10,
  restore_best_weights=True,
  mode='max'
  )
  optimizer = keras.optimizers.Adam(learning_rate=0.0001)

  model.compile(
      optimizer=optimizer,
      loss='binary_crossentropy',
      metrics=[            'accuracy',
          keras.metrics.AUC(name='auc'),
          keras.metrics.Precision(name='precision'),
          keras.metrics.Recall(name='recall')
      ]
      )

  history = model.fit(
        x_train,
        y_train,
        epochs=125,
        batch_size=64,
        validation_data=(x_test, y_test),
        class_weight=class_weights_dict
        #callbacks=[early_stop]
    )

  model.evaluate(x_test, y_test)
  model.save("canaryv14.keras")

main()




{0: np.float64(0.5352149145520456), 1: np.float64(7.599264705882353)}
Epoch 1/125
33/33 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.6430 - auc: 0.6980 - loss: 0.8343 - precision: 0.1141 - recall: 0.6544 - val_accuracy: 0.5126 - val_auc: 0.6673 - val_loss: 1.2589 - val_precision: 0.0933 - val_recall: 0.7353
Epoch 2/125
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6381 - auc: 0.6682 - loss: 0.8696 - precision: 0.1047 - recall: 0.5956 - val_accuracy: 0.6054 - val_auc: 0.6724 - val_loss: 1.0859 - val_precision: 0.1028 - val_recall: 0.6471
Epoch 3/125
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6449 - auc: 0.7058 - loss: 0.8061 - precision: 0.1176 - recall: 0.6765 - val_accuracy: 0.6518 - val_auc: 0.6626 - val_loss: 0.9840 - val_precision: 0.1117 - val_recall: 0.6176
Epoch 4/125
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6405 - auc: 0.6979 - loss: 0.8088 - precision: 0.1134 - recall: 0.6544 - val_accuracy: 0.6654 - val_auc: 0.6568 - val_loss: 0.9292 - val_prec

In [187]:
import numpy as np
from sklearn.metrics import recall_score, accuracy_score


y_probs = model.predict(x_test)


for t in [0.5, 0.45, 0.4, 0.35, 0.3, 0.25]:
    y_pred = (y_probs > t).astype(int)

    rec = recall_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    print(f"Threshold: {t:.2f} | Recall: {rec:.2f} | Accuracy: {acc:.2f}")





17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Threshold: 0.50 | Recall: 0.00 | Accuracy: 0.90
Threshold: 0.45 | Recall: 0.18 | Accuracy: 0.72
Threshold: 0.40 | Recall: 0.50 | Accuracy: 0.27
Threshold: 0.35 | Recall: 0.79 | Accuracy: 0.11
Threshold: 0.30 | Recall: 0.97 | Accuracy: 0.08
Threshold: 0.25 | Recall: 1.00 | Accuracy: 0.07


In [ ]:

for threshold in np.arange(0.30, 0.40, 0.02):
    y_pred = (y_probs > threshold).astype(int)
    rec = recall_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    print(f"Threshold: {threshold:.3f} | Recall: {rec:.2f} | Accuracy: {acc:.2f}")


In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report


treemodel = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=30,
    use_label_encoder=False;
)

treemodel.fit(x_train, y_train)

y_pred = treemodel.predict(x_test)
print(classification_report(y_test, y_pred))
y_probs = treemodel.predict_proba(x_test)[:, 1]

for threshold in [0.5, 0.45, 0.4, 0.35, 0.3]:
    y_pred = (y_probs > threshold).astype(int)
    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    print(f"Threshold {threshold:.2f} | Accuracy: {acc:.2f} | Recall: {rec:.2f}")

treemodel.save_model('treemodel.json')




/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [23:40:24] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              precision    recall  f1-score   support

           0       0.96      0.80      0.87       483
           1       0.16      0.56      0.25        34

    accuracy                           0.78       517
   macro avg       0.56      0.68      0.56       517
weighted avg       0.91      0.78      0.83       517

Threshold 0.50 | Accuracy: 0.78 | Recall: 0.56
Threshold 0.45 | Accuracy: 0.74 | Recall: 0.59
Threshold 0.40 | Accuracy: 0.68 | Recall: 0.65
Threshold 0.35 | Accuracy: 0.63 | Recall: 0.68
Threshold 0.30 | Accuracy: 0.56 | Recall: 0.71


In [186]:
import sklearn
print(sklearn.__version__)

1.6.1
